In [1]:
import zmq
import json
import numpy as np


In [7]:
kingkong_address = "tcp://kingkong.ph.ic.ac.uk"
port = 5000
context = zmq.Context()
socket = context.socket(zmq.DEALER)
socket.connect(f"{kingkong_address}:{port}")
# --- timeouts in milliseconds ---
socket.setsockopt(zmq.RCVTIMEO, 1000)   # 1 sec receive timeout
socket.setsockopt(zmq.SNDTIMEO, 1000)   # 1 sec send timeout

array = np.array([1, 2, 3, 4, 5])

header = {
    "shape": array.shape,
    "dtype": str(array.dtype)
}

socket.send_multipart([
    json.dumps(header).encode(),
    array.tobytes()   # raw contiguous buffer
], copy=False)


In [ ]:
try:
    socket.send_multipart(
        [json.dumps(header).encode(), memoryview(array)],
        copy=False
    )

    reply = socket.recv_json()
    print("Reply:", reply)

except zmq.Again:
    print("Request timed out")
    
    # IMPORTANT: REQ socket is now in a bad state
    socket.close(linger=0)
    socket = context.socket(zmq.REQ)
    socket.connect(f"{kingkong_address}:{port}")

TypeError: Socket.send() missing 1 required positional argument: 'data'

AttributeError: 'list' object has no attribute 'tobytes'